# Shakespeare Text Generation with an LSTM

Train a word-level next-token language model, evaluate it on held-out text, and generate continuations from several seed phrases. The notebook includes an optional small architecture/sequence-length comparison.

The source is Project Gutenberg's public-domain (in the United States) edition of Shakespeare's *Complete Works*: [ebook 100](https://www.gutenberg.org/ebooks/100). Run cells from top to bottom. The first run downloads the text and trains the model; runtime varies substantially by hardware. TensorFlow is required (`pip install tensorflow`).

## 1. Load and preprocess the text

The loader prefers a local `shakespeare.txt` file in the notebook directory and otherwise downloads the Gutenberg plain-text edition. Gutenberg's wrapper is removed when its standard markers are present. Preprocessing lowercases the text, removes punctuation, and splits on whitespace. A contiguous token cap keeps the example practical on a laptop; increase it for more training data if memory and runtime allow.

In [1]:
from collections import Counter
from pathlib import Path
import re
import urllib.request
import numpy as np
import tensorflow as tf

SEED = 17
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)

TensorFlow: 2.21.0


In [2]:
DATA_URL = 'https://www.gutenberg.org/cache/epub/100/pg100.txt'
LOCAL_TEXT = Path('shakespeare.txt')

if LOCAL_TEXT.exists():
    raw_text = LOCAL_TEXT.read_text(encoding='utf-8', errors='replace')
    print(f'Loaded local dataset: {LOCAL_TEXT.resolve()}')
else:
    request = urllib.request.Request(DATA_URL, headers={'User-Agent': 'LSTM-text-generation-notebook/1.0'})
    with urllib.request.urlopen(request, timeout=60) as response:
        raw_text = response.read().decode('utf-8', errors='replace')
    print('Downloaded Project Gutenberg ebook 100.')

start_marker = '*** START OF THE PROJECT GUTENBERG EBOOK'
end_marker = '*** END OF THE PROJECT GUTENBERG EBOOK'
start = raw_text.find(start_marker)
if start >= 0:
    newline = raw_text.find('\n', start)
    raw_text = raw_text[newline + 1:] if newline >= 0 else raw_text
end = raw_text.find(end_marker)
if end >= 0:
    raw_text = raw_text[:end]

clean_text = re.sub(r'[^a-z0-9\s]', ' ', raw_text.lower())
tokens = clean_text.split()
MAX_TOKENS = 100_000
tokens = tokens[:MAX_TOKENS]
if len(tokens) < 5_000:
    raise ValueError(f'Only {len(tokens)} tokens were loaded; check the dataset download or local text file.')
print(f'Usable tokens: {len(tokens):,}')
print('Preview:', ' '.join(tokens[:40]))

Downloaded Project Gutenberg ebook 100.
Usable tokens: 100,000
Preview: the complete works of william shakespeare by william shakespeare contents the sonnets all s well that ends well the tragedy of antony and cleopatra as you like it the comedy of errors the tragedy of coriolanus cymbeline the tragedy of


## 2. Build next-token examples

Reserve the final portion of the token stream for validation so evaluation reflects later, unseen text. Vocabulary is learned from the training section only; rare and unseen words map to `<unk>`. Each example contains `SEQUENCE_LENGTH` words and predicts the following word. Padding is reserved for generation prompts, not inserted into training examples.

In [3]:
SEQUENCE_LENGTH = 20
MAX_VOCAB_SIZE = 8_000
VALIDATION_FRACTION = 0.10

split_at = int(len(tokens) * (1 - VALIDATION_FRACTION))
train_tokens = tokens[:split_at]
validation_tokens = tokens[split_at:]

PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
most_common = [word for word, _ in Counter(train_tokens).most_common(MAX_VOCAB_SIZE - 2)]
vocabulary = [PAD_TOKEN, UNK_TOKEN, *most_common]
token_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_token = np.asarray(vocabulary, dtype=object)

def encode(words):
    unknown_id = token_to_id[UNK_TOKEN]
    return np.fromiter((token_to_id.get(word, unknown_id) for word in words), dtype=np.int32)

def make_examples(words, sequence_length):
    encoded = encode(words)
    windows = np.lib.stride_tricks.sliding_window_view(encoded, sequence_length + 1)
    return windows[:, :-1], windows[:, -1]

X_train, y_train = make_examples(train_tokens, SEQUENCE_LENGTH)
X_validation, y_validation = make_examples(validation_tokens, SEQUENCE_LENGTH)
print(f'Vocabulary: {len(vocabulary):,}')
print(f'Training examples: {len(X_train):,}; validation examples: {len(X_validation):,}')
print('Example input:', ' '.join(id_to_token[X_train[0]]))
print('Target:', id_to_token[y_train[0]])

Vocabulary: 7,947
Training examples: 89,980; validation examples: 9,980
Example input: the complete works of william shakespeare by william shakespeare contents the sonnets all s well that ends well the tragedy
Target: of


## 3. Define and train the model

The embedding maps word IDs to learned dense vectors. The LSTM summarizes the context, and the softmax layer assigns a probability to every vocabulary item. Sparse categorical crossentropy avoids one-hot encoding the target. Early stopping restores the best validation-loss weights; a checkpoint is also saved as `best_lstm_text.keras`. Adjust the epoch count, token cap, and model size to suit your machine.

In [4]:
BATCH_SIZE = 128
EMBEDDING_DIM = 96
LSTM_UNITS = 128
EPOCHS = 10

def make_dataset(inputs, targets, batch_size=BATCH_SIZE, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))
    if shuffle:
        dataset = dataset.shuffle(min(len(inputs), 20_000), seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(X_train, y_train, shuffle=True)
validation_dataset = make_dataset(X_validation, y_validation)

def build_model(sequence_length, vocabulary_size, units=LSTM_UNITS, layers=1, embedding_dim=EMBEDDING_DIM):
    inputs = tf.keras.Input(shape=(sequence_length,), dtype=tf.int32)
    x = tf.keras.layers.Embedding(vocabulary_size, embedding_dim, mask_zero=True)(inputs)
    for layer_index in range(layers):
        x = tf.keras.layers.LSTM(units, return_sequences=(layer_index < layers - 1))(x)
    outputs = tf.keras.layers.Dense(vocabulary_size, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
    return model

model = build_model(SEQUENCE_LENGTH, len(vocabulary))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 96)    │    762,912 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 20)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │    115,200 │ embedding[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 7947)      │  1,025,163 │ lstm[0][0]        │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,903,275 (7.26 MB)

 Trainable params: 1,903,275 (7.26 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_lstm_text.keras', monitor='val_loss', save_best_only=True),
]
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
)

validation_loss, validation_accuracy = model.evaluate(validation_dataset, verbose=0)
print(f'Validation loss: {validation_loss:.4f}')
print(f'Validation next-word accuracy: {validation_accuracy:.2%}')
print(f'Validation perplexity: {np.exp(validation_loss):.2f}')

Epoch 1/10


c:\Users\talav\OneDrive\Desktop\Jenya\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


703/703 ━━━━━━━━━━━━━━━━━━━━ 175s 223ms/step - loss: 6.7629 - sparse_categorical_accuracy: 0.0322 - val_loss: 6.8076 - val_sparse_categorical_accuracy: 0.0423
Epoch 2/10
703/703 ━━━━━━━━━━━━━━━━━━━━ 147s 209ms/step - loss: 6.3599 - sparse_categorical_accuracy: 0.0462 - val_loss: 6.7834 - val_sparse_categorical_accuracy: 0.0551
Epoch 3/10
703/703 ━━━━━━━━━━━━━━━━━━━━ 140s 199ms/step - loss: 6.1642 - sparse_categorical_accuracy: 0.0594 - val_loss: 6.7712 - val_sparse_categorical_accuracy: 0.0617
Epoch 4/10
703/703 ━━━━━━━━━━━━━━━━━━━━ 149s 212ms/step - loss: 5.9771 - sparse_categorical_accuracy: 0.0746 - val_loss: 6.7325 - val_sparse_categorical_accuracy: 0.0725
Epoch 5/10
703/703 ━━━━━━━━━━━━━━━━━━━━ 143s 202ms/step - loss: 5.7833 - sparse_categorical_accuracy: 0.0879 - val_loss: 6.7297 - val_sparse_categorical_accuracy: 0.0742
Epoch 6/10
703/703 ━━━━━━━━━━━━━━━━━━━━ 230s 327ms/step - loss: 5.6057 - sparse_categorical_accuracy: 0.0978 - val_loss: 6.7609 - val_sparse_categorical_accuracy

## 4. Generate text from seed phrases

At each step, sample a word from the model's temperature-scaled distribution and feed it back into the context. Lower temperatures usually favor safer, more repetitive continuations; higher temperatures increase variety and incoherence. The output cells below will contain real samples after the training and generation cells have run.

In [6]:
def generate_text(seed_text, model, number_of_words=60, temperature=0.8):
    if temperature <= 0:
        raise ValueError('temperature must be greater than zero')
    seed_words = re.sub(r'[^a-z0-9\s]', ' ', seed_text.lower()).split()
    if not seed_words:
        raise ValueError('Provide at least one seed word.')
    generated_ids = [token_to_id.get(word, token_to_id[UNK_TOKEN]) for word in seed_words]
    context = generated_ids[-SEQUENCE_LENGTH:]
    context = [token_to_id[PAD_TOKEN]] * (SEQUENCE_LENGTH - len(context)) + context
    for _ in range(number_of_words):
        probabilities = model(np.asarray([context], dtype=np.int32), training=False)[0]
        log_probabilities = tf.math.log(tf.maximum(probabilities, 1e-8)) / temperature
        next_id = int(tf.random.categorical(log_probabilities[None, :], num_samples=1)[0, 0].numpy())
        generated_ids.append(next_id)
        context = (context + [next_id])[-SEQUENCE_LENGTH:]
    return ' '.join(str(id_to_token[index]) for index in generated_ids)

seed_phrases = [
    'to be or not to be',
    'shall i compare thee',
    'the course of true love',
]
for phrase in seed_phrases:
    print(f'\nSeed: {phrase}')
    print(generate_text(phrase, model, number_of_words=60, temperature=0.8))


Seed: to be or not to be
to be or not to be speak of our villain of her a old creature i health you keep the eyes in their soil almost like the wind of the priest and colour caesar i ll have your song too clock s more than i am this where they live so not not have more weather rosalind he was no earth in the forest of fast

Seed: shall i compare thee
shall i compare thee what and my sake no shall never tell me if i am about in his fact when they have a fool but she are thy honour i will be my blood your uncle the good vermilion of the of the effigies evil the fool may our caitiff no pretty will be two cicatrice therefore put i ll not it as

Seed: the course of true love
the course of true love shall be dead is i be old beg by the duke fellow yet he are his was orlando s the bell parolles hours thou be beyond i am a antique er you had with me the forest of coriolanus enjoyed a hat by death frederick it is rosalind duke and thou hast i love thee pour my heart must be


## 5. Optional bonus: compare context and depth

Set `RUN_BONUS_EXPERIMENTS = True` to run a small controlled comparison: a shallow LSTM at two context lengths and a two-layer LSTM at the longer context. Each trial uses the same token split and a short epoch budget to keep the experiment manageable. Compare validation loss/perplexity and inspect generations; the numerical metric does not fully capture literary coherence. Longer contexts and deeper models can help capture dependencies, but also cost more compute and can overfit. Results are produced only when the experiment is run.

In [8]:
RUN_BONUS_EXPERIMENTS = True
BONUS_EPOCHS = 3

if RUN_BONUS_EXPERIMENTS:
    results = []
    configurations = [(12, 1), (24, 1), (24, 2)]
    for sequence_length, layer_count in configurations:
        x_train, y_train = make_examples(train_tokens, sequence_length)
        x_validation, y_validation = make_examples(validation_tokens, sequence_length)
        trial = build_model(sequence_length, len(vocabulary), layers=layer_count)
        trial.fit(
            make_dataset(x_train, y_train, shuffle=True),
            validation_data=make_dataset(x_validation, y_validation),
            epochs=BONUS_EPOCHS,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)],
            verbose=0,
        )
        loss, accuracy = trial.evaluate(make_dataset(x_validation, y_validation), verbose=0)
        perplexity = float(np.exp(loss))
        results.append({'sequence_length': sequence_length, 'layers': layer_count, 'val_loss': loss, 'val_accuracy': accuracy, 'val_perplexity': perplexity})
        print(f'seq={sequence_length:2d}, layers={layer_count}: val_loss={loss:.4f}, accuracy={accuracy:.2%}, perplexity={perplexity:.2f}')
    print('Compare these validation results with the main model; trials use fewer epochs and are not a definitive quality ranking.')
else:
    print('Bonus experiments skipped. Set RUN_BONUS_EXPERIMENTS = True and rerun this cell to compare configurations.')

c:\Users\talav\OneDrive\Desktop\Jenya\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


seq=12, layers=1: val_loss=6.7365, accuracy=6.16%, perplexity=842.61
seq=24, layers=1: val_loss=6.7498, accuracy=6.04%, perplexity=853.92
seq=24, layers=2: val_loss=6.7874, accuracy=3.62%, perplexity=886.56
Compare these validation results with the main model; trials use fewer epochs and are not a definitive quality ranking.


## Notes and limitations

- The model learns local word patterns, not Shakespeare's full semantics, dramatic structure, or meter; fluent-looking stretches may be nonsensical.
- Training on only the first `MAX_TOKENS` tokens and limiting vocabulary make this demonstration faster, but reduce coverage. Increase both cautiously for richer results.
- A chronological split tests continuation on later parts of the text, which may include different plays and vocabulary than the training section.
- Generated examples are stochastic. Rerun generation with different temperatures or seeds to compare outputs.

Dataset source: [Project Gutenberg, The Complete Works of William Shakespeare, ebook 100](https://www.gutenberg.org/ebooks/100). If the download is unavailable, place a UTF-8 plain-text copy named `shakespeare.txt` next to this notebook and rerun the loading cell.